# PySpark DataFrame DSL — Part 11: Date & Time Functions, Transformations, and Formatting

> **Goal:** Learn Spark date/timestamp types, datetime patterns, parsing, extracting components, arithmetic, duration calculations, formatting, time zones, data quality, and practical ETL use cases. The notes follow the previous parts: definitions, setup, examples, explanations, large-data examples, troubleshooting, assignments, and quick reference.

## Table of Contents
1. Setup and sample data
2. Date and timestamp fundamentals
3. Spark datetime format patterns
4. Parsing and casting
5. Extracting date/time components
6. Date arithmetic
7. Timestamp arithmetic and durations
8. Truncation and formatting
9. Current date/time and Unix timestamps
10. Time zones
11. Nulls and invalid dates
12. Filtering ranges and incremental ETL
13. Bronze → Silver → Gold pipeline
14. Large-data example
15. Common mistakes and performance
16. Assignments
17. Quick reference

---

# 1. Setup and Sample Data

```python
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("PySpark_Date_Time_Functions")
    .getOrCreate()
)
print("Spark version:", spark.version)
print("Session time zone:", spark.conf.get("spark.sql.session.timeZone", "not-set"))
```

Set a business time zone deliberately before processing timestamps where possible:

```python
spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")
```

Changing this setting can affect timestamp parsing, display, and conversion. Avoid changing it halfway through a pipeline.

```python
orders_data = [
    (1001, "C101", "2026-01-05", "2026-01-05 09:15:30", "COMPLETED", 65000.0),
    (1002, "C102", "2026-01-06", "2026-01-06 14:20:00", "COMPLETED", 32000.0),
    (1003, "C103", "2026-01-07", "2026-01-07 23:55:10", "PENDING", 12000.0),
    (1004, "C101", "2026-01-08", "2026-01-08 00:05:45", "COMPLETED", 2500.0),
    (1005, "C104", "2026-01-09", "2026-01-09 18:40:00", "CANCELLED", 7500.0),
    (1006, "C105", "2026-01-10", "2026-01-10 11:05:15", "COMPLETED", 18000.0),
    (1007, "C106", "2026-01-11", "2026-01-11 16:30:00", "PENDING", 22000.0),
    (1008, "C107", "2026-01-12", "2026-01-12 08:00:00", "COMPLETED", 1800.0),
]
columns = ["order_id", "customer_id", "order_date_raw", "order_timestamp_raw", "status", "amount"]
df_orders_raw = spark.createDataFrame(orders_data, columns)
df_orders_raw.show(truncate=False)
df_orders_raw.printSchema()

df_orders = (
    df_orders_raw
    .withColumn("order_date", F.to_date("order_date_raw", "yyyy-MM-dd"))
    .withColumn("order_timestamp", F.to_timestamp("order_timestamp_raw", "yyyy-MM-dd HH:mm:ss"))
)
df_orders.printSchema()
df_orders.show(truncate=False)
```

# 2. Date and Timestamp Fundamentals

| Type | Meaning | Example |
|---|---|---|
| `StringType` | Text; not automatically a date | `"2026-01-05"` |
| `DateType` | Calendar date, no time-of-day | `2026-01-05` |
| `TimestampType` | Timestamp value; display/conversion depends on session zone and Spark timestamp semantics | `2026-01-05 09:15:30` |
| Unix epoch seconds | Seconds from `1970-01-01 00:00:00 UTC` | `1767604530` |

Parse raw text early. Typed date/time values make comparisons and arithmetic safer. String sorting is not safe for arbitrary date formats, and timestamp display may differ from the original input when zones are involved.

# 3. Spark Datetime Format Patterns

Spark patterns are **not** Python `strftime` patterns.

| Pattern | Meaning | Example |
|---|---|---|
| `yyyy` | Four-digit year | `2026` |
| `yy` | Two-digit year | `26` |
| `M` / `MM` | Month number | `3` / `03` |
| `MMM` / `MMMM` | Abbreviated/full month name | `Mar` / `March` |
| `d` / `dd` | Day of month | `5` / `05` |
| `D` | Day of year | `64` |
| `E` / `EEE` | Day name / abbreviated weekday | `Saturday` / `Sat` |
| `u` | Day number, Monday=1 through Sunday=7 in Spark datetime patterns | `6` |
| `HH` | 24-hour hour (00–23) | `18` |
| `hh` | 12-hour hour (01–12) | `06` |
| `mm` | Minute | `45` |
| `ss` | Second | `30` |
| `S` | Fractional-second digit(s) | `123` |
| `a` | AM/PM | `PM` |
| `XXX` | Common zone offset pattern | `+05:30` |
| `V` | Zone ID in supported contexts | `Asia/Kolkata` |

**Case matters:** `MM` = month, `mm` = minute; `HH` = 24-hour clock, `hh` = 12-hour clock.

| Input | Pattern |
|---|---|
| `2026-03-15` | `yyyy-MM-dd` |
| `15/03/2026` | `dd/MM/yyyy` |
| `03-15-2026` | `MM-dd-yyyy` |
| `20260315` | `yyyyMMdd` |
| `2026-03-15 18:45:30` | `yyyy-MM-dd HH:mm:ss` |
| `15/03/2026 06:45 PM` | `dd/MM/yyyy hh:mm a` |
| `2026-03-15T18:45:30` | `yyyy-MM-dd'T'HH:mm:ss` |
| `2026-03-15T18:45:30+05:30` | `yyyy-MM-dd'T'HH:mm:ssXXX` |

A literal `T` must be quoted in a pattern: `'T'`.

```python
mixed = spark.createDataFrame(
    [(1, "2026-03-14"), (2, "15/03/2026"), (3, "03-16-2026"), (4, "20260317")],
    ["id", "raw_date"]
)
parsed = mixed.withColumn(
    "parsed_date",
    F.when(F.col("raw_date").rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date("raw_date", "yyyy-MM-dd"))
     .when(F.col("raw_date").rlike(r"^\d{2}/\d{2}/\d{4}$"), F.to_date("raw_date", "dd/MM/yyyy"))
     .when(F.col("raw_date").rlike(r"^\d{2}-\d{2}-\d{4}$"), F.to_date("raw_date", "MM-dd-yyyy"))
     .when(F.col("raw_date").rlike(r"^\d{8}$"), F.to_date("raw_date", "yyyyMMdd"))
)
parsed.show(truncate=False)
```

The input shape helps choose a parser but does not prove a calendar date is valid.

# 4. Parsing and Casting

## 4.1 `to_date(col, format)`

**Definition:** Parses a compatible string/expression into `DateType`. `format` is an optional Spark datetime pattern.

```python
F.to_date("order_date_raw", "yyyy-MM-dd")
```

### Five examples

```python
# 1. Parse ISO date
x = df_orders_raw.withColumn("d", F.to_date("order_date_raw", "yyyy-MM-dd"))

# 2. Parse day/month/year
spark.createDataFrame([(1, "27/10/2026")], ["id", "raw"]).withColumn(
    "d", F.to_date("raw", "dd/MM/yyyy")
).show()

# 3. Parse compact date
spark.createDataFrame([(1, "20261027")], ["id", "raw"]).withColumn(
    "d", F.to_date("raw", "yyyyMMdd")
).show()

# 4. Timestamp to date
x.select("order_timestamp", F.to_date("order_timestamp").alias("date_only")).show()

# 5. Flag parse failures
x.withColumn(
    "parse_failed", F.col("order_date_raw").isNotNull() & F.col("d").isNull()
).show()
```

Malformed input behavior depends on Spark version, ANSI mode, and parser settings. Do not assume every invalid input becomes null in every environment.

## 4.2 `to_timestamp(col, format)`

**Definition:** Parses a string/expression into a timestamp. Supply a pattern for non-default input formats.

```python
F.to_timestamp("order_timestamp_raw", "yyyy-MM-dd HH:mm:ss")
```

### Five examples

```python
# 1. Standard timestamp
x = df_orders_raw.withColumn(
    "ts", F.to_timestamp("order_timestamp_raw", "yyyy-MM-dd HH:mm:ss")
)

# 2. 12-hour time with AM/PM
spark.createDataFrame([(1, "27/10/2026 06:45 PM")], ["id", "raw"]).withColumn(
    "ts", F.to_timestamp("raw", "dd/MM/yyyy hh:mm a")
).show(truncate=False)

# 3. ISO string with literal T
spark.createDataFrame([(1, "2026-10-27T18:45:30")], ["id", "raw"]).withColumn(
    "ts", F.to_timestamp("raw", "yyyy-MM-dd'T'HH:mm:ss")
).show(truncate=False)

# 4. Timestamp with offset
spark.createDataFrame([(1, "2026-10-27T18:45:30+05:30")], ["id", "raw"]).withColumn(
    "ts", F.to_timestamp("raw", "yyyy-MM-dd'T'HH:mm:ssXXX")
).show(truncate=False)

# 5. Flag invalid timestamps
x.withColumn(
    "parse_failed", F.col("order_timestamp_raw").isNotNull() & F.col("ts").isNull()
).show()
```

Timestamp parsing and display depend on session time zone and Spark timestamp configuration.

## 4.3 `cast()`

**Definition:** Converts a column expression to a supported Spark SQL type. Casting is convenient for standard forms; explicit parsing patterns are safer for nonstandard strings.

```python
# 1. Standard ISO string to date
x = df_orders_raw.withColumn("d", F.col("order_date_raw").cast("date"))
# 2. Numeric text to integer
spark.createDataFrame([("101",)], ["raw"]).select(F.col("raw").cast("int")).show()
# 3. Numeric text to decimal
spark.createDataFrame([("123.45",)], ["raw"]).select(F.col("raw").cast("decimal(10,2)")).show()
# 4. Date to string
x.select(F.col("d").cast("string").alias("date_text")).show()
# 5. Timestamp to date
x.withColumn("ts", F.to_timestamp("order_timestamp_raw", "yyyy-MM-dd HH:mm:ss")) \
 .withColumn("date_only", F.col("ts").cast("date")).show()
```

# 5. Extracting Date/Time Components

## 5.1 `year()`, `quarter()`, `month()`, `dayofmonth()`

**Definition:** Extract calendar components from a date or timestamp expression.

```python
# 1. Year
x.select("order_date", F.year("order_date").alias("year")).show()
# 2. Quarter (1–4)
x.select("order_date", F.quarter("order_date").alias("quarter")).show()
# 3. Month number
x.select("order_date", F.month("order_date").alias("month")).show()
# 4. Day of month
x.select("order_date", F.dayofmonth("order_date").alias("day")).show()
# 5. Year-month key
x.withColumn("year_month", F.date_format("order_date", "yyyy-MM")).select("order_id", "year_month").show()
```

## 5.2 `dayofweek()`, `weekday()`, `dayofyear()`

**Definition:** Extract weekday/day-of-year values. Numbering differs by function:
- `dayofweek()`: Sunday=1 through Saturday=7.
- `weekday()`: Monday=0 through Sunday=6.
- `dayofyear()`: 1 through 365/366.

```python
# 1. Sunday-based weekday number
x.select("order_date", F.dayofweek("order_date").alias("dow")).show()
# 2. Monday-based weekday number
x.select("order_date", F.weekday("order_date").alias("weekday")).show()
# 3. Day of year
x.select("order_date", F.dayofyear("order_date").alias("doy")).show()
# 4. Monday–Friday filter
x.filter(F.weekday("order_date").between(0, 4)).show()
# 5. Weekend flag
x.withColumn("is_weekend", F.weekday("order_date").isin(5, 6)).show()
```

## 5.3 `hour()`, `minute()`, `second()`

**Definition:** Extract time-of-day components from a timestamp.

```python
# 1. Hour
x.select(F.hour("order_timestamp").alias("hour")).show()
# 2. Minute
x.select(F.minute("order_timestamp").alias("minute")).show()
# 3. Second
x.select(F.second("order_timestamp").alias("second")).show()
# 4. Classify time period
x.withColumn("period",
    F.when(F.hour("order_timestamp").between(5, 11), "MORNING")
     .when(F.hour("order_timestamp").between(12, 16), "AFTERNOON")
     .when(F.hour("order_timestamp").between(17, 20), "EVENING")
     .otherwise("NIGHT")
).show()
# 5. Filter business hours
x.filter(F.hour("order_timestamp").between(9, 17)).show()
```

## 5.4 `weekofyear()`

**Definition:** Returns the week number for a date. Year-boundary rules can be surprising; confirm ISO week-year requirements for your Spark version.

```python
# 1. Week number
x.select("order_date", F.weekofyear("order_date").alias("week")).show()
# 2. Group by year/week
x.withColumn("year", F.year("order_date")).withColumn("week", F.weekofyear("order_date")).groupBy("year", "week").count().show()
# 3. Filter week 2
x.filter(F.weekofyear("order_date") == 2).show()
# 4. Year-week label
x.withColumn("year_week", F.concat_ws("-W", F.year("order_date").cast("string"), F.lpad(F.weekofyear("order_date").cast("string"), 2, "0"))).show()
# 5. Inspect dates around boundaries
x.select("order_date", F.weekofyear("order_date")).orderBy("order_date").show()
```

# 6. Date Arithmetic

## 6.1 `date_add()` and `date_sub()`

**Definition:** Add/subtract a number of calendar days from a date.

```python
# 1. Add 7 days
x.withColumn("delivery_date", F.date_add("order_date", 7)).show()
# 2. Subtract 7 days
x.withColumn("previous_week", F.date_sub("order_date", 7)).show()
# 3. Dynamic offset
x.withColumn("future_date", F.date_add("order_date", F.col("order_id") % 5)).show()
# 4. Follow-up date after 30 days
x.withColumn("follow_up_date", F.date_add("order_date", 30)).show()
# 5. Filter by derived cutoff
x.filter(F.date_add("order_date", 7) < F.lit("2026-01-20").cast("date")).show()
```

## 6.2 `datediff(end, start)`

**Definition:** Returns the calendar-day difference from `start` to `end`.

```python
# 1. Days since order
x.withColumn("age_days", F.datediff(F.current_date(), "order_date")).show()
# 2. Fixed date difference
spark.range(1).select(F.datediff(F.lit("2026-10-27").cast("date"), F.lit("2026-10-01").cast("date")).alias("days")).show()
# 3. SLA date difference
x.withColumn("delivery_date", F.date_add("order_date", 5)).withColumn("sla_days", F.datediff("delivery_date", "order_date")).show()
# 4. Older than 10 days
x.withColumn("age_days", F.datediff(F.current_date(), "order_date")).filter(F.col("age_days") > 10).show()
# 5. Two date columns
spark.createDataFrame([("2026-10-10", "2026-10-01")], ["end_raw", "start_raw"]).select(
    F.datediff(F.to_date("end_raw"), F.to_date("start_raw")).alias("days_between")
).show()
```

Do not use `datediff()` to calculate exact elapsed hours; it measures calendar-day boundaries.

## 6.3 `add_months()`

**Definition:** Adds a positive or negative number of calendar months. Month-end behavior is calendar-aware.

```python
# 1. Next month
x.withColumn("next_month", F.add_months("order_date", 1)).show()
# 2. Three months earlier
x.withColumn("three_months_before", F.add_months("order_date", -3)).show()
# 3. Dynamic month offset
x.withColumn("future_date", F.add_months("order_date", F.col("order_id") % 6)).show()
# 4. Renewal date in 12 months
x.withColumn("renewal_date", F.add_months("order_date", 12)).show()
# 5. Prior-month comparison date
x.withColumn("prior_month", F.add_months("order_date", -1)).show()
```

## 6.4 `months_between(end, start)`

**Definition:** Returns fractional months between two dates/timestamps. Same day-of-month or both month-end cases can produce whole months; otherwise Spark applies its documented month calculation rules.

```python
# 1. Fixed dates
spark.range(1).select(F.months_between(F.lit("2026-10-27").cast("date"), F.lit("2026-01-27").cast("date")).alias("months")).show()
# 2. Months since order
x.withColumn("months_since", F.months_between(F.current_date(), "order_date")).show()
# 3. Floor to whole months only if the business rule permits
x.withColumn("whole_months", F.floor(F.months_between(F.current_date(), "order_date")).cast("int")).show()
# 4. Renewal difference
x.withColumn("renewal", F.add_months("order_date", 12)).withColumn("months", F.months_between("renewal", "order_date")).show()
# 5. Round for reporting
x.withColumn("months_rounded", F.round(F.months_between(F.current_date(), "order_date"), 2)).show()
```

Do not treat fractional months as a universal billing rule.

## 6.5 `last_day()` and `next_day()`

**Definition:** `last_day()` returns the last calendar day of the month. `next_day(date, weekday)` returns the first named weekday strictly after the date.

```python
# 1. Month end
x.withColumn("month_end", F.last_day("order_date")).show()
# 2. First day of next month
x.withColumn("next_month_start", F.date_add(F.last_day("order_date"), 1)).show()
# 3. Next Monday
x.withColumn("next_monday", F.next_day("order_date", "Mon")).show()
# 4. Next Friday
x.withColumn("next_friday", F.next_day("order_date", "Fri")).show()
# 5. Days until month end
x.withColumn("days_to_month_end", F.datediff(F.last_day("order_date"), "order_date")).show()
```

# 7. Timestamp Arithmetic and Durations

## 7.1 `unix_timestamp()` and `from_unixtime()`

**Definition:** `unix_timestamp()` converts compatible input to epoch seconds. `from_unixtime()` converts epoch seconds to a formatted **string**, not a timestamp type.

```python
# 1. Timestamp to epoch seconds
x.withColumn("epoch_seconds", F.unix_timestamp("order_timestamp")).show()
# 2. Parse raw text and convert
x.withColumn("epoch_seconds", F.unix_timestamp("order_timestamp_raw", "yyyy-MM-dd HH:mm:ss")).show()
# 3. Epoch seconds to default formatted string
x.withColumn("epoch", F.unix_timestamp("order_timestamp")).withColumn("epoch_text", F.from_unixtime("epoch")).show(truncate=False)
# 4. Epoch seconds to custom string
x.withColumn("epoch", F.unix_timestamp("order_timestamp")).withColumn("formatted", F.from_unixtime("epoch", "yyyy-MM-dd HH:mm:ss")).show(truncate=False)
# 5. Format as year-month
x.withColumn("epoch", F.unix_timestamp("order_timestamp")).withColumn("year_month", F.from_unixtime("epoch", "yyyy-MM")).show()
```

Epoch conversion uses timestamp/time-zone semantics. Keep typed timestamps for computation and format strings for presentation/export.

## 7.2 `timestamp_seconds()`, `timestamp_millis()`, `timestamp_micros()`

**Definition:** Convert epoch seconds, milliseconds, or microseconds to timestamp values in Spark versions that support these functions.

```python
# 1. Seconds
spark.createDataFrame([(1793124000,)], ["epoch_seconds"]).select(F.timestamp_seconds("epoch_seconds").alias("ts")).show()
# 2. Milliseconds
spark.createDataFrame([(1793124000000,)], ["epoch_millis"]).select(F.timestamp_millis("epoch_millis").alias("ts")).show()
# 3. Microseconds
spark.createDataFrame([(1793124000000000,)], ["epoch_micros"]).select(F.timestamp_micros("epoch_micros").alias("ts")).show()
# 4. Keep source and converted timestamp
spark.createDataFrame([(1793124000,)], ["epoch_seconds"]).withColumn("ts", F.timestamp_seconds("epoch_seconds")).show()
# 5. Filter converted values
spark.createDataFrame([(1793124000,), (1793210400,)], ["epoch_seconds"]).withColumn(
    "ts", F.timestamp_seconds("epoch_seconds")
).filter(F.col("ts") >= F.lit("2026-10-27 00:00:00").cast("timestamp")).show()
```

Check availability in your Spark version. For older versions, a supported `from_unixtime()`/cast approach may be used, but validate zone semantics.

## 7.3 Calculate elapsed time

For elapsed seconds, subtract epoch seconds. This loses sub-second precision; use a supported timestamp-difference or microsecond-aware strategy if needed.

```python
events = spark.createDataFrame([
    (1, "2026-10-27 09:00:00", "2026-10-27 10:30:00"),
    (2, "2026-10-27 23:30:00", "2026-10-28 00:15:00"),
], ["event_id", "start_raw", "end_raw"]).withColumn(
    "start_ts", F.to_timestamp("start_raw", "yyyy-MM-dd HH:mm:ss")
).withColumn("end_ts", F.to_timestamp("end_raw", "yyyy-MM-dd HH:mm:ss"))

# 1. Elapsed seconds
events.withColumn("elapsed_seconds", F.unix_timestamp("end_ts") - F.unix_timestamp("start_ts")).show()
# 2. Elapsed minutes
events.withColumn("elapsed_minutes", (F.unix_timestamp("end_ts") - F.unix_timestamp("start_ts")) / 60.0).show()
# 3. Elapsed hours
events.withColumn("elapsed_hours", (F.unix_timestamp("end_ts") - F.unix_timestamp("start_ts")) / 3600.0).show()
# 4. SLA breach over 60 minutes
events.withColumn("minutes", (F.unix_timestamp("end_ts") - F.unix_timestamp("start_ts")) / 60.0).withColumn("sla_breached", F.col("minutes") > 60).show()
# 5. Invalid negative duration
events.withColumn("seconds", F.unix_timestamp("end_ts") - F.unix_timestamp("start_ts")).withColumn("invalid", F.col("seconds") < 0).show()
```

# 8. Truncation and Formatting

## 8.1 `trunc(date, format)` vs `date_trunc(format, timestamp)`

**Definition:** `trunc()` truncates a date to a larger calendar unit and returns a date. `date_trunc()` truncates a timestamp to a unit and returns a timestamp.

```python
# 1. Month start (date)
x.withColumn("month_start", F.trunc("order_date", "month")).show()
# 2. Year start
x.withColumn("year_start", F.trunc("order_date", "year")).show()
# 3. Timestamp hour bucket
x.withColumn("hour_bucket", F.date_trunc("hour", "order_timestamp")).show()
# 4. Timestamp day bucket
x.withColumn("timestamp_day", F.date_trunc("day", "order_timestamp")).show()
# 5. Aggregate by month
x.withColumn("month_start", F.trunc("order_date", "month")).groupBy("month_start").agg(F.sum("amount").alias("revenue")).show()
```

Supported units commonly include year, quarter, month, week, day, hour, minute, and second, depending on function/version.

## 8.2 `date_format(col, format)`

**Definition:** Formats a date/timestamp as a string. It does not return a date type.

```python
# 1. ISO date string
x.withColumn("date_text", F.date_format("order_date", "yyyy-MM-dd")).show()
# 2. Day/month/year
x.withColumn("date_text", F.date_format("order_date", "dd/MM/yyyy")).show()
# 3. Timestamp string
x.withColumn("timestamp_text", F.date_format("order_timestamp", "yyyy-MM-dd HH:mm:ss")).show(truncate=False)
# 4. Full month name
x.withColumn("month_name", F.date_format("order_date", "MMMM")).show()
# 5. Revenue by year-month
x.withColumn("year_month", F.date_format("order_date", "yyyy-MM")).groupBy("year_month").agg(F.sum("amount").alias("revenue")).show()
```

# 9. Current Date/Time and Time Zones

## 9.1 `current_date()` and `current_timestamp()`

**Definition:** Return the current date/timestamp at the start of query evaluation.

```python
# 1. Processing date
x.withColumn("processing_date", F.current_date()).show()
# 2. Load timestamp
x.withColumn("processed_at", F.current_timestamp()).show(truncate=False)
# 3. Age in days
x.withColumn("age_days", F.datediff(F.current_date(), "order_date")).show()
# 4. Rows dated today
x.filter(F.col("order_date") == F.current_date()).show()
# 5. Audit column
x.withColumn("etl_loaded_at", F.current_timestamp()).select("order_id", "etl_loaded_at").show(truncate=False)
```

Separate actions/queries may evaluate these functions at different times.

## 9.2 `to_utc_timestamp()` and `from_utc_timestamp()`

**Definition:** Helpers for interpreting/converting timestamp values between UTC and a specified time zone. Their behavior depends on timestamp type, session zone, and whether input already contains an offset. Avoid double-converting already-normalized values.

```python
local_df = spark.createDataFrame([("2026-10-27 18:00:00",)], ["raw"]).withColumn(
    "local_ts", F.to_timestamp("raw", "yyyy-MM-dd HH:mm:ss")
)
# 1. Interpret/convert a Kolkata local timestamp to UTC semantics
local_df.withColumn("utc_ts", F.to_utc_timestamp("local_ts", "Asia/Kolkata")).show(truncate=False)

utc_df = spark.createDataFrame([("2026-01-15 12:00:00",)], ["raw"]).withColumn(
    "utc_ts", F.to_timestamp("raw", "yyyy-MM-dd HH:mm:ss")
)
# 2. Display in New York zone
utc_df.withColumn("new_york_ts", F.from_utc_timestamp("utc_ts", "America/New_York")).show(truncate=False)
# 3. Display in India zone
utc_df.withColumn("india_ts", F.from_utc_timestamp("utc_ts", "Asia/Kolkata")).show(truncate=False)
# 4. Format after conversion
utc_df.withColumn("india_ts", F.from_utc_timestamp("utc_ts", "Asia/Kolkata")).withColumn(
    "india_text", F.date_format("india_ts", "yyyy-MM-dd HH:mm:ss")
).show(truncate=False)
# 5. Compare zone renderings
utc_df.withColumn("india_ts", F.from_utc_timestamp("utc_ts", "Asia/Kolkata")).withColumn(
    "ny_ts", F.from_utc_timestamp("utc_ts", "America/New_York")
).select("utc_ts", "india_ts", "ny_ts").show(truncate=False)
```

Best practices:
- Know whether input means UTC, local wall-clock time, or a value with an explicit offset.
- Set a consistent session time zone.
- Convert for display only when appropriate.
- Test midnight, month/year boundaries, leap days, and daylight-saving transitions for relevant zones.
- Timestamp display is affected by Spark's timestamp type/configuration and `spark.sql.session.timeZone`.

# 10. Nulls and Invalid Dates

Malformed input may throw or become null depending on Spark version, ANSI mode, and parser settings. Use a tolerant parsing function where supported if the pipeline must handle malformed values without failing.

```python
bad_dates = spark.createDataFrame([
    (1, "2026-02-15"), (2, "2026-02-30"), (3, "not-a-date"), (4, None)
], ["id", "raw_date"])
parsed_bad = bad_dates.withColumn("parsed_date", F.to_date("raw_date", "yyyy-MM-dd"))
parsed_bad.show(truncate=False)
```

### Five data-quality examples

```python
parsed = df_orders_raw.withColumn("parsed_date", F.to_date("order_date_raw", "yyyy-MM-dd")).withColumn(
    "parse_status",
    F.when(F.col("order_date_raw").isNull(), "MISSING")
     .when(F.col("parsed_date").isNull(), "INVALID")
     .otherwise("VALID")
)
# 1. Find missing strings
parsed.filter(F.col("order_date_raw").isNull()).show()
# 2. Find invalid parse results
parsed.filter(F.col("order_date_raw").isNotNull() & F.col("parsed_date").isNull()).show()
# 3. Count by status
parsed.groupBy("parse_status").count().show()
# 4. Keep invalid rows for quarantine
invalid_rows = parsed.filter(F.col("parse_status") != "VALID")
invalid_rows.show(truncate=False)
# 5. Add a fallback only if the business rule permits it
parsed.withColumn("report_date", F.coalesce(F.col("parsed_date"), F.lit("1900-01-01").cast("date"))).show()
```

Sentinel dates can be mistaken for real dates. Prefer explicit parse status and quarantine where possible.

# 11. Filtering Date Ranges and Incremental ETL

## Inclusive date range

```python
x.filter(F.col("order_date").between(F.lit("2026-01-05").cast("date"), F.lit("2026-01-10").cast("date"))).show()
```

`between()` includes both endpoints.

## Half-open timestamp range

```python
start_ts = F.lit("2026-01-05 00:00:00").cast("timestamp")
end_ts = F.lit("2026-01-06 00:00:00").cast("timestamp")
x.filter((F.col("order_timestamp") >= start_ts) & (F.col("order_timestamp") < end_ts)).show()
```

Half-open intervals avoid missing fractional seconds at the end of a day.

## Last seven calendar days

```python
x.filter(F.col("order_date") >= F.date_sub(F.current_date(), 7)).show()
```

Clarify whether the requirement means calendar dates or the last 168 elapsed hours.

## Watermark example

```python
watermark = "2026-01-08 00:00:00"
incremental_df = x.filter(F.col("order_timestamp") >= F.to_timestamp(F.lit(watermark)))
incremental_df.show()
```

Production watermarks need reliable persistence, deduplication, and a late-arrival policy. A lookback window may be safer than strictly filtering after the last timestamp.

# 12. Practical Bronze → Silver → Gold ETL Pipeline

```python
bronze = df_orders_raw

silver = (
    bronze
    .withColumn("order_date", F.to_date("order_date_raw", "yyyy-MM-dd"))
    .withColumn("order_timestamp", F.to_timestamp("order_timestamp_raw", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("date_parse_status",
        F.when(F.col("order_date_raw").isNull(), "MISSING")
         .when(F.col("order_date").isNull(), "INVALID")
         .otherwise("VALID"))
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .withColumn("year_month", F.date_format("order_date", "yyyy-MM"))
    .withColumn("order_week", F.weekofyear("order_date"))
    .withColumn("order_day", F.dayofmonth("order_date"))
)

silver_valid = silver.filter((F.col("date_parse_status") == "VALID") & F.col("order_timestamp").isNotNull())
silver_invalid = silver.filter((F.col("date_parse_status") != "VALID") | F.col("order_timestamp").isNull())
print("Valid:", silver_valid.count(), "Invalid:", silver_invalid.count())

gold_daily_revenue = (
    silver_valid.filter(F.col("status") == "COMPLETED")
    .groupBy("order_date")
    .agg(F.count("*").alias("completed_order_count"),
         F.sum("amount").alias("daily_revenue"),
         F.avg("amount").alias("average_order_amount"))
    .orderBy("order_date")
)
gold_daily_revenue.show()

gold_hourly_metrics = (
    silver_valid.withColumn("hour_bucket", F.date_trunc("hour", "order_timestamp"))
    .groupBy("hour_bucket")
    .agg(F.count("*").alias("event_count"), F.sum("amount").alias("total_amount"))
    .orderBy("hour_bucket")
)
gold_hourly_metrics.show(truncate=False)
```

Production considerations: validate currency precision, duplicate IDs, timestamp meaning, late-arriving events, and business time zone before using results for financial reporting.

# 13. Large-Data Example

Use Spark-native generation rather than building millions of Python objects on the driver.

```python
large_events = (
    spark.range(0, 1_000_000)
    .withColumnRenamed("id", "event_id")
    .withColumn("event_timestamp", F.expr("timestamp'2026-01-01 00:00:00'") + F.expr("INTERVAL 1 SECOND") * F.col("event_id"))
    .withColumn("customer_id", F.col("event_id") % 100_000)
    .withColumn("amount", (F.col("event_id") % 1000 + 1).cast("double"))
    .withColumn("event_type",
        F.when(F.col("event_id") % 4 == 0, "CLICK")
         .when(F.col("event_id") % 4 == 1, "VIEW")
         .when(F.col("event_id") % 4 == 2, "PURCHASE")
         .otherwise("LOGIN"))
)
```

If your Spark version does not support interval multiplication in that form, use a version-supported timestamp expression.

```python
# A. Aggregate by date, hour, and event type
large_hourly = large_events.withColumn("event_date", F.to_date("event_timestamp")).withColumn(
    "event_hour", F.hour("event_timestamp")
).groupBy("event_date", "event_hour", "event_type").agg(
    F.count("*").alias("event_count"), F.sum("amount").alias("total_amount")
)
large_hourly.show()

# B. Filter one half-open day window
large_window = large_events.filter(
    (F.col("event_timestamp") >= F.lit("2026-01-02 00:00:00").cast("timestamp")) &
    (F.col("event_timestamp") < F.lit("2026-01-03 00:00:00").cast("timestamp"))
)
print("Events in window:", large_window.count())

# C. Compare plans
large_events.filter(F.to_date("event_timestamp") == F.lit("2026-01-02").cast("date")).explain("formatted")
large_events.filter((F.col("event_timestamp") >= F.lit("2026-01-02 00:00:00").cast("timestamp")) &
                    (F.col("event_timestamp") < F.lit("2026-01-03 00:00:00").cast("timestamp")).explain("formatted")
```

For partitioned storage, filter directly on partition columns when possible and inspect the physical plan to confirm partition pruning.

# 14. Common Mistakes and Performance Notes

1. **Month vs minute:** `MM` is month; `mm` is minute.
2. **Python formatting vs Spark patterns:** Spark uses `yyyy-MM-dd`, not `%Y-%m-%d`.
3. **Formatted output is text:** `date_format()` returns string. Keep typed dates/timestamps for calculations.
4. **Implicit parsing:** Explicitly parse nonstandard input formats.
5. **`datediff()` is not elapsed hours:** It calculates calendar-day difference.
6. **Ignoring time zones:** Define whether inputs represent UTC, local wall-clock time, or offset-aware instants.
7. **Fixed 24-hour assumption:** In some zones, daylight-saving transitions mean a local calendar day is not always 24 elapsed hours.
8. **`months_between()` is not universal billing logic:** Use the business contract's rules.
9. **Inclusive timestamp end:** Prefer `>= start` and `< next_period_start` instead of `<= 23:59:59`.
10. **Silent invalid data:** Keep parse status and quarantine invalid rows.

Performance checklist:
- Parse raw strings once and use typed values downstream.
- Filter on typed boundaries and preserve partition-pruning opportunities.
- Avoid repeatedly formatting timestamps to strings during calculations.
- Set the session time zone deliberately.
- Use `explain("formatted")` and Spark UI to inspect plans and scan metrics.
- Test leap days, month ends, year boundaries, invalid strings, nulls, and relevant zone transitions.

# 15. Practice Assignments

## Beginner
1. Parse `order_date_raw` as `DateType`.
2. Parse `order_timestamp_raw` as `TimestampType`.
3. Extract year, quarter, month, day of month, and weekday.
4. Format dates as `dd/MM/yyyy`.
5. Add seven days to each order date.
6. Find the last day of each order's month.
7. Calculate days since each order.
8. Extract hour and minute.
9. Create a weekend flag.
10. Filter an inclusive date range.

## Intermediate
11. Parse `dd/MM/yyyy`, `MM-dd-yyyy`, and `yyyyMMdd` formats.
12. Add a `year_month` reporting key.
13. Calculate months between an order and renewal date.
14. Calculate elapsed seconds/minutes between event start/end.
15. Build a daily revenue report for completed orders.
16. Build an hourly event-count report.
17. Flag missing and invalid dates separately.
18. Convert epoch seconds using functions supported by your Spark version.
19. Demonstrate the effect of session time zone on timestamp display/conversion.
20. Filter a daily ingestion window using half-open timestamp boundaries.

## Advanced
21. Build Bronze → Silver → Gold processing with parse-status tracking and invalid-row quarantine.
22. Generate one million events and aggregate by date/hour/type.
23. Compare `trunc()` and `date_trunc()` result types.
24. Compare `datediff()` with elapsed seconds across midnight.
25. Test leap day and month-end behavior with `add_months()` and `last_day()`.
26. Parse values with UTC offsets and explain display semantics.
27. Design partition-pruning-friendly date filters.
28. Design a watermark policy that handles late arrivals and duplicates.

# 16. Quick Reference

## Parsing and formatting

| Function | Purpose | Result |
|---|---|---|
| `to_date(col, format)` | Parse date | `DateType` |
| `to_timestamp(col, format)` | Parse timestamp | Timestamp type; zone/configuration matters |
| `cast("date")` | Cast compatible value to date | Pattern control is limited |
| `date_format(col, format)` | Format date/timestamp | String |
| `unix_timestamp()` | Convert to epoch seconds | Numeric seconds |
| `from_unixtime()` | Epoch seconds to formatted text | String |
| `timestamp_seconds()` | Epoch seconds to timestamp | Version-dependent availability |
| `timestamp_millis()` | Epoch milliseconds to timestamp | Version-dependent availability |
| `timestamp_micros()` | Epoch microseconds to timestamp | Version-dependent availability |

## Extractors

| Function | Purpose |
|---|---|
| `year()` / `quarter()` / `month()` | Calendar year, quarter, month |
| `dayofmonth()` | Day in month |
| `dayofweek()` | Sunday=1 through Saturday=7 |
| `weekday()` | Monday=0 through Sunday=6 |
| `dayofyear()` | Day in year |
| `weekofyear()` | Week number |
| `hour()` / `minute()` / `second()` | Time components |
| `current_date()` / `current_timestamp()` | Current query date/time |

## Arithmetic and bucketing

| Function | Purpose |
|---|---|
| `date_add()` / `date_sub()` | Add/subtract calendar days |
| `datediff(end, start)` | Calendar-day difference |
| `add_months()` | Add/subtract months |
| `months_between()` | Fractional month difference |
| `last_day()` | Last day of month |
| `next_day()` | Next named weekday after date |
| `trunc(date, format)` | Truncate date to larger unit |
| `date_trunc(format, timestamp)` | Truncate timestamp to unit |
| `to_utc_timestamp()` / `from_utc_timestamp()` | Time-zone conversion helpers |

## Final Takeaway

- Parse raw strings into the correct Spark types early.
- Use patterns such as `yyyy-MM-dd HH:mm:ss`; remember `MM` is month and `mm` is minute.
- Keep typed values for computation and format strings for presentation.
- Distinguish calendar-day differences from elapsed durations.
- Set time-zone semantics deliberately and avoid double conversion.
- Validate malformed values and retain parse-status information.
- Test leap days, month ends, year boundaries, and time-zone transitions relevant to your data.
